# Silver: Soft Delete / Restore Application

Handles logical deletion and restoration of job postings.

## Operations

### Soft Delete
- Mark records as deleted without physical removal
- Preserve history for audit and potential restoration
- Reasons: expired, removed from source, DQ failure, duplicate

### Restore
- Reactivate previously deleted records
- Update restoration timestamp and reason
- Clear soft delete flags

## Use Cases
- Jobs removed from source APIs (aged out)
- Quarantined records being reinstated
- Manual administrative actions

In [0]:
dbutils.widgets.dropdown("operation", "delete", ["delete", "restore"], "Operation Type")
dbutils.widgets.text("job_ids", "", "Comma-separated enterprise_job_ids (empty for batch)")
dbutils.widgets.text("reason", "", "Reason for operation")
dbutils.widgets.text("source_filter", "", "Filter by source name")

operation = dbutils.widgets.get("operation")
job_ids = dbutils.widgets.get("job_ids").strip()
reason = dbutils.widgets.get("reason").strip()
source_filter = dbutils.widgets.get("source_filter").strip()

print(f"Operation: {operation.upper()}")
print(f"Job IDs: {job_ids if job_ids else 'Batch mode'}")
print(f"Reason: {reason if reason else 'Not specified'}")
print(f"Source Filter: {source_filter if source_filter else 'All sources'}")

In [0]:
import json
from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"

run_timestamp = F.current_timestamp()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Run ID: {run_id}")

In [0]:
# Build query based on operation
if operation == "delete":
    # Select active jobs for deletion
    base_query = f"""
    SELECT enterprise_job_id, source_name, title_raw, company_name_raw
    FROM {SILVER_SCHEMA}.silver_jobs_current
    WHERE is_active = true AND soft_delete_flag = false
    """
else:  # restore
    # Select soft-deleted jobs for restoration
    base_query = f"""
    SELECT enterprise_job_id, source_name, title_raw, company_name_raw
    FROM {SILVER_SCHEMA}.silver_jobs_current
    WHERE soft_delete_flag = true
    """

# Apply filters
if job_ids:
    id_list = [f"'{jid.strip()}'" for jid in job_ids.split(",")]
    base_query += f" AND enterprise_job_id IN ({','.join(id_list)})"

if source_filter:
    base_query += f" AND source_name = '{source_filter}'"

target_jobs_df = spark.sql(base_query)
target_count = target_jobs_df.count()

print(f"Jobs selected for {operation}: {target_count}")

if target_count == 0:
    dbutils.notebook.exit({"status": "success", "message": f"No jobs to {operation}"})

display(target_jobs_df.limit(10))

In [0]:
if operation == "delete":
    # Prepare update
    update_df = target_jobs_df.select("enterprise_job_id").withColumn(
        "soft_delete_flag", F.lit(True)
    ).withColumn(
        "soft_delete_reason", F.lit(reason if reason else "Manual soft delete")
    ).withColumn(
        "is_active", F.lit(False)
    ).withColumn(
        "soft_deleted_at", run_timestamp
    )
    
    # Merge to current table
    delta_current = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_jobs_current")
    
    delta_current.alias("cur").merge(
        update_df.alias("upd"),
        "cur.enterprise_job_id = upd.enterprise_job_id"
    ).whenMatchedUpdate(
        set={
            "soft_delete_flag": "upd.soft_delete_flag",
            "soft_delete_reason": "upd.soft_delete_reason",
            "is_active": "upd.is_active",
            "updated_at": "upd.soft_deleted_at"
        }
    ).execute()
    
    print(f"Soft deleted {target_count} jobs")
    
    # Log to changes table
    spark.sql(f"""
    INSERT INTO {SILVER_SCHEMA}.silver_job_changes
    SELECT 
        uuid() as change_id,
        enterprise_job_id,
        source_name,
        'SOFT_DELETE' as change_type,
        CAST(NULL AS ARRAY<STRING>) as changed_columns,
        NULL as old_value_json,
        NULL as new_value_json,
        current_timestamp() as change_timestamp,
        '{run_id}' as batch_id
    FROM ({base_query})
    """)
    
    result_message = f"Successfully soft deleted {target_count} jobs"

In [0]:
if operation == "restore":
    # Prepare update
    update_df = target_jobs_df.select("enterprise_job_id").withColumn(
        "soft_delete_flag", F.lit(False)
    ).withColumn(
        "soft_delete_reason", F.lit(None).cast("string")
    ).withColumn(
        "is_active", F.lit(True)
    ).withColumn(
        "restored_at", run_timestamp
    ).withColumn(
        "restoration_reason", F.lit(reason if reason else "Manual restore")
    )
    
    # Merge to current table
    delta_current = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_jobs_current")
    
    delta_current.alias("cur").merge(
        update_df.alias("upd"),
        "cur.enterprise_job_id = upd.enterprise_job_id"
    ).whenMatchedUpdate(
        set={
            "soft_delete_flag": "upd.soft_delete_flag",
            "soft_delete_reason": "upd.soft_delete_reason",
            "is_active": "upd.is_active",
            "updated_at": "upd.restored_at"
        }
    ).execute()
    
    print(f"Restored {target_count} jobs")
    
    # Log to changes table
    spark.sql(f"""
    INSERT INTO {SILVER_SCHEMA}.silver_job_changes
    SELECT 
        uuid() as change_id,
        enterprise_job_id,
        source_name,
        'RESTORE' as change_type,
        CAST(NULL AS ARRAY<STRING>) as changed_columns,
        NULL as old_value_json,
        NULL as new_value_json,
        current_timestamp() as change_timestamp,
        '{run_id}' as batch_id
    FROM ({base_query})
    """)
    
    result_message = f"Successfully restored {target_count} jobs"

In [0]:
result = {
    "status": "success",
    "run_id": run_id,
    "operation": operation,
    "records_affected": target_count,
    "message": result_message
}

print("\n=== Operation Summary ===")
print(json.dumps(result, indent=2))

dbutils.notebook.exit(json.dumps(result))